In [ ]:
import requests
import pandas as pd

base = "https://hancock.research.fau.eu/include/get_data.php?tab="

clinical = requests.get(base + "clinical_data").json()
slides = requests.get(base + "primary_tumor_slides").json()

In [ ]:
df_clin = pd.DataFrame(clinical)
df_slides = pd.DataFrame(slides)

In [ ]:
dup_ids = df_slides.loc[
    df_slides["patient_id"].duplicated(keep=False),
    "patient_id"
].unique()
total_slide_duplicates = df_slides["patient_id"].duplicated().sum()
print(dup_ids)

['36' '239' '251' '277' '428' '489' '589' '732']


In [ ]:
print(df_clin.columns)
print(df_slides.columns)

Index(['clinical_id', 'patient_id', 'year_of_initial_diagnosis',
       'age_at_initial_diagnosis', 'sex', 'smoking_status',
       'primarily_metastasis', 'survival_status', 'survival_status_with_cause',
       'days_to_last_information', 'first_treatment_intent',
       'first_treatment_modality', 'days_to_first_treatment',
       'adjuvant_treatment_intent', 'adjuvant_radiotherapy',
       'adjuvant_radiotherapy_modality', 'adjuvant_systemic_therapy',
       'adjuvant_systemic_therapy_modality', 'adjuvant_radiochemotherapy',
       'recurrence', 'days_to_recurrence', 'progress_1', 'days_to_progress_1',
       'progress_2', 'days_to_progress_2', 'metastasis_1_locations',
       'days_to_metastasis_1', 'metastasis_2_locations',
       'days_to_metastasis_2', 'metastasis_3_locations',
       'days_to_metastasis_3', 'metastasis_4_locations',
       'days_to_metastasis_4'],
      dtype='object')
Index(['wsi_id', 'patient_id', 'file_path', 'thumbnail_path', 'location',
       'clinical_id

In [ ]:
clin_ids = set(df_clin["patient_id"])
slide_ids = set(df_slides["patient_id"])

In [ ]:
missing_in_slides = clin_ids - slide_ids

print(len(missing_in_slides))
print(list(missing_in_slides))

62
['356', '96', '348', '341', '173', '204', '129', '197', '282', '742', '299', '266', '300', '497', '272', '571', '754', '186', '535', '508', '686', '107', '438', '265', '27', '132', '268', '448', '526', '633', '541', '446', '537', '97', '143', '150', '710', '515', '329', '580', '659', '748', '629', '332', '302', '67', '241', '305', '596', '199', '91', '381', '54', '395', '13', '729', '382', '76', '89', '440', '598', '759']


To ensure data integrity, we remove patients with duplicate primary tumor slide entries, keeping only a single slide per patient to prevent label leakage during training. We further exclude patients with no associated primary tumor slide, since histopathology is a required modality in our framework and its absence makes a patient unusable across all imaging-dependent components. We validate cohort completeness by cross-referencing patient ids between the clinical and slide tables prior to downstream preprocessing.